# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 9


## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM, sau model lingvistic mare, este un program de inteligență artificială antrenat pe o cantitate uriașă de text, capabil să înțeleagă, să genereze și să răspundă la limbajul uman într-un mod coerent și relevant. Aceste modele pot realiza sarcini variate precum traducerea, rezumarea textelor, scrierea de conținut creativ și răspunsuri la întrebări.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe cantități masive de text pentru a înțelege, genera și manipula limbajul uman într-un mod coerent și contextual. Aceste modele pot îndeplini o gamă largă de sarcini lingvistice, de la răspunsuri la întrebări și scrierea de conținut, la traducere și rezumare.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, politica românească a fost marcată de alternanța la guvernare și de apariția unor noi formațiuni politice, influențând dinamica parlamentară. De asemenea, s-au accentuat dezbaterile și implementarea unor reforme legislative în domenii precum justiția și administrația publică.

--- Gemini 2.5 Flash ---
Ultimii cinci ani au fost marcați de o succesiune de guverne și coaliții, culminând cu formarea unei alianțe largi între partidele tradiționale PNL și PSD. Această nouă formulă a inclus un acord de rotație a prim-ministrului, stabilizând peisajul politic după o perioadă de instabilitate.

--- OpenRouter Free ---
În ultimii cinci ani, România a trecut printr-o serie de schimbări de guvernare, cu multiplele formări de guvern și alianțe politice, inclusiv guvernul PNL-USR din 2020 și alianța PSD-PNL-USR din 2024. Reformele în justiție și relația cu UE au fost central, cu accent pe accesarea fondurilor europene și lupta împotriva corupției.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, cinic
Emoție dominantă: Frustrare, neîncredere
Țintă principală: Clasa politică
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, cinic.
Emoție dominantă: Frustrare.
Țintă principală: Clasa politică.
Populism: da

--- OpenRouter Free ---
**Ton:** Agresiv și acuzator.  
**Emoție dominantă:** Răbdare și frustrare.  
**Țintă principală:** Critică față de corupția politică și impunitia, cu o percepție de ignorare față de nevoile populației.  
**Populism:** Da. Comentariul folosește un dichotomie "popor vs. elite" și expresie de desiluzie față de sistemul politic.


**Emoție dominantă:** Răbdare și frustrare.   - mai degraba nerabdare...

## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o frustrare generală față de politicieni, acuzându-i de corupție și ignorarea voinței poporului, un discurs tipic populist.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă furie și dezamăgire față de clasa politică, acuzând-o de corupție și de ignorarea poporului, folosind o retorică populistă prin contrastul 'oameni simpli' vs. 'politicieni'."}

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'frica', 'tinta_principala': 'insatisfacție publică', 'populism': True, 'explicatie_scurta': 'Comentariul expresează o desconfianță profundă față de politicienii, acuzându-i de corupție, iar critică responsabilitatea sistemului politic pentru greșeli financiare și sociala ale poporului. Se

--- OpenRouter Free --- O cam ia razna la raspuns...nici nu o termina.

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [12]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.3,0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Acest proces poate influența stabilitatea guvernamentală și direcția politicilor publice pe termen scurt și mediu.

temperature=0.3:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea de noi alegeri, ceea ce implică o perioadă de incertitudine politică și o posibilă reconfigurare a forțelor politice. Acest proces poate influența stabilitatea guvernării și procesul decizional pe termen scurt.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea unor noi alegeri, permițând astfel o reconfigurare a peisajului politic. Acest proces poate genera incertitudine pe termen scurt și poate influența agenda legislativă și deciziil

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? (Imi este foarte greu de inteles cuvantul "adnotare") | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da   | da  | da | nu | Toate raspunsurile transmit acelasi lucru si nu vad prea multa creativitate chiar si la tempreture mai inalt. |
| Gemini 2.5 Flash | da  | da  | da | nu | Ca si la versiunea anterioara, raspunsurile sunt destul de consecvente si transmit aceasi idee in estenta. |
| OpenRouter Free | nu | parțial | nu | da  | Singurul raspuns valid pare sa fie varianta 2, unde nu am surprins greseli. In rest sunt greseli gramaticale si de limba, astfel nu putem sa ne bazam pe raspunsuri. |
### Decizie
**Model principal ales:**  Gemini 2.5 Flash
**Model de rezervă:**  Gemini 2.5 Flash Light
**Temperature recomandată:**  0.3
**De ce am ales acest model?**  Nu sunt sigura ca pot oferi un raspuns cat de cat obiectiv, biasul meu personal avand un efect asupra raspunsului. Am ales varianta 2 din cauza raspunsului legat de "percepția publică asupra stabilității și legitimității instituțiilor". Perceptia mea este influentata de aceasta mentiune, impartasind neincrederea in institutii. Deci pentru mine raspunsul este cel mai nuantat si detaliat, stabil si robust pentru a fi folosit in continuare. (Nu stiu ce inseamna adnotare...am cautat in dex si exemple si nu am reusit sa cuprind sensul cuvantului)

**Raspunsurile mele din tabel s-au bazat pe prima rulare, unde varianta 3 - OpenRouter Free a inclus greseli si a pus litere in Coreana in raspuns. 

Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [13]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.3

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales